In [1]:
# Required Libraries
import os
import re
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, random_split, Dataset, Subset
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from collections import Counter, defaultdict
from PIL import Image
import optuna  # Hyperparameter optimization

# Set device (select primary GPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Function to wrap module in DataParallel if more than one GPU is available
def maybe_data_parallel(model):
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    return model

# Configurations
BATCH_SIZE = 32  # Lowered batch size for memory savings
EPOCHS = 10     # Number of epochs for training
NUM_FOLDS = 3   # Number of folds for cross-validation
DATA_DIR = "/kaggle/input/d/sabarnasaha1/openaimer-2025-track1/OpenAImer2025_Image_Classification/OpenAImer"
TEST_DIR = os.path.join(DATA_DIR, "test")
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Data Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

augmented_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Raw Dataset for Custom Augmentations (returns a raw PIL image)
class RawImageDataset(Dataset):
    def __init__(self, subset: Subset):
        self.subset = subset  # subset from random_split or KFold
        self.dataset = subset.dataset  # original ImageFolder dataset
        self.indices = subset.indices

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        path, label = self.dataset.samples[actual_idx]
        img = Image.open(path).convert("RGB")
        return img, label

    def __len__(self):
        return len(self.indices)

# Dataset Loading for full training data
base_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=None)
num_classes = len(base_dataset.classes)
le = LabelEncoder().fit(base_dataset.classes)

# Test Dataset Definition
class TestDataset(Dataset):
    def __init__(self, test_dir, transform):
        self.filenames = [f for f in os.listdir(test_dir) if f.endswith('.jpg')]
        self.filepaths = [os.path.join(test_dir, f) for f in self.filenames]
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        img = self.transform(img)
        file = self.filenames[idx]
        img_id = int(re.findall(r'(\d+)', file)[0])
        return img, img_id

# Contrastive Loss Function
def supervised_contrastive_loss(features, labels, temperature=0.07):
    labels = labels.contiguous().view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)
    contrast_count = features.shape[0]
    anchor_dot_contrast = torch.div(torch.matmul(features, features.T), temperature)
    logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
    logits = anchor_dot_contrast - logits_max.detach()
    exp_logits = torch.exp(logits) * (1 - torch.eye(contrast_count).to(device))
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)
    mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)
    return -mean_log_prob_pos.mean()

# Projection Head Module
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=128):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, out_dim)
        )

    def forward(self, x):
        return self.proj(x)

# Model Wrapper: Supports resnet50, vgg16, efficientnet_b0
def get_model(name):
    base_model = getattr(models, name)(weights="IMAGENET1K_V1")
    if name.startswith("vgg"):
        feat_dim = base_model.classifier[-1].in_features
        base_model.classifier = nn.Sequential(*list(base_model.classifier.children())[:-1])
        encoder = nn.Sequential(base_model.features, nn.Flatten(), base_model.classifier)
    elif name.startswith("efficientnet"):
        feat_dim = base_model.classifier[1].in_features
        base_model.classifier = nn.Sequential(*list(base_model.classifier.children())[:-1])
        encoder = nn.Sequential(base_model.features, nn.AdaptiveAvgPool2d(1), nn.Flatten(), base_model.classifier)
    else:  # resnet50
        feat_dim = base_model.fc.in_features
        base_model.fc = nn.Identity()
        encoder = base_model
    encoder = maybe_data_parallel(encoder)
    projector = maybe_data_parallel(ProjectionHead(feat_dim))
    return encoder.to(device), projector.to(device), feat_dim

# ----------------------------------
# Hyperparameter Optimization with Optuna using AMP
# ----------------------------------
def objective(trial, model_name, train_loader, val_loader):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    optimizer_choice = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    
    encoder, projector, feat_dim = get_model(model_name)
    classifier = nn.Linear(feat_dim, num_classes).to(device)
    classifier = maybe_data_parallel(classifier)
    
    if optimizer_choice == "Adam":
        optimizer = torch.optim.Adam(list(encoder.parameters()) + list(projector.parameters()), lr=lr)
    else:
        optimizer = torch.optim.SGD(list(encoder.parameters()) + list(projector.parameters()), lr=lr, momentum=0.9)
    
    criterion_ce = nn.CrossEntropyLoss()
    total_loss = 0
    scaler = torch.cuda.amp.GradScaler()
    
    # Run for 2 epochs for hyperparameter speed-up
    for epoch in range(2):
        encoder.train(); projector.train()
        running_loss = 0
        for imgs, labels in train_loader:
            with torch.cuda.amp.autocast():
                imgs1 = torch.stack([augmented_transform(img) for img in imgs])
                imgs2 = torch.stack([augmented_transform(img) for img in imgs])
                imgs_all = torch.cat([imgs1, imgs2], dim=0).to(device)
                labels_all = torch.tensor(labels * 2).to(device)
                feats = projector(encoder(imgs_all))
                loss = supervised_contrastive_loss(F.normalize(feats, dim=1), labels_all)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
        total_loss += running_loss / len(train_loader)
        torch.cuda.empty_cache()
    return total_loss / 2

# ----------------------------------
# Training with K-Fold Cross Validation using AMP
# ----------------------------------
def train_cv(model_name):
    kfold = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
    fold_val_scores = []
    models_cv = []
    indices = list(range(len(base_dataset)))
    for fold, (train_idx, val_idx) in enumerate(kfold.split(indices)):
        print(f"Fold {fold+1}/{NUM_FOLDS}")
        train_subset = Subset(base_dataset, train_idx)
        val_subset = Subset(base_dataset, val_idx)
        train_ds = RawImageDataset(train_subset)
        val_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=transform)
        val_ds.samples = [val_subset.dataset.samples[i] for i in val_subset.indices]
        train_loader_cv = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
        val_loader_cv = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
        
        study = optuna.create_study(direction="minimize")
        study.optimize(lambda trial: objective(trial, model_name, train_loader_cv, val_loader_cv), n_trials=3)
        best_params = study.best_params
        best_lr = best_params["lr"]
        best_opt = best_params["optimizer"]
        print(f"Best Params for fold {fold+1}: {best_params}")
        
        encoder, projector, feat_dim = get_model(model_name)
        classifier = nn.Linear(feat_dim, num_classes).to(device)
        classifier = maybe_data_parallel(classifier)
        if best_opt == "Adam":
            optimizer = torch.optim.Adam(list(encoder.parameters()) + list(projector.parameters()), lr=best_lr)
        else:
            optimizer = torch.optim.SGD(list(encoder.parameters()) + list(projector.parameters()), lr=best_lr, momentum=0.9)
        criterion_ce = nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, verbose=True)
        scaler = torch.cuda.amp.GradScaler()
        for epoch in range(EPOCHS):
            encoder.train(); projector.train()
            running_loss = 0
            for imgs, labels in tqdm(train_loader_cv, desc=f"{model_name} Fold {fold+1} Epoch {epoch+1}"):
                with torch.cuda.amp.autocast():
                    imgs1 = torch.stack([augmented_transform(img) for img in imgs])
                    imgs2 = torch.stack([augmented_transform(img) for img in imgs])
                    imgs_all = torch.cat([imgs1, imgs2], dim=0).to(device)
                    labels_all = torch.tensor(labels * 2).to(device)
                    feats = projector(encoder(imgs_all))
                    loss = supervised_contrastive_loss(F.normalize(feats, dim=1), labels_all)
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                running_loss += loss.item()
            train_loss_fold = running_loss / len(train_loader_cv)
            scheduler.step(train_loss_fold)
            print(f"Fold {fold+1} Epoch {epoch+1} Loss: {train_loss_fold:.4f}")
        encoder.eval(); classifier.train()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for imgs, labels in val_loader_cv:
                imgs, labels = imgs.to(device), labels.to(device)
                feats = encoder(imgs)
                logits = classifier(feats)
                loss = criterion_ce(logits, labels)
                val_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                correct += (preds == labels).sum().item()
                total += len(labels)
        val_acc = correct / total
        print(f"Fold {fold+1} Validation Accuracy: {val_acc:.4f}")
        fold_val_scores.append(val_acc)
        models_cv.append((encoder, classifier))
        torch.cuda.empty_cache()
    return np.mean(fold_val_scores), models_cv

# ----------------------------------
# Test-Time Augmentation (TTA) using AMP
# ----------------------------------
def predict_tta(encoder, classifier, num_tta=5):
    testset = TestDataset(TEST_DIR, transform)
    testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)
    predictions, probs = {}, {}
    encoder.eval(); classifier.eval()
    
    with torch.no_grad():
        for imgs, ids in testloader:
            imgs = imgs.to(device)
            with torch.cuda.amp.autocast():
                logits = classifier(encoder(imgs))
            tta_logits = [logits]
            for _ in range(num_tta - 1):
                imgs_aug = []
                for img in imgs:
                    pil_img = transforms.ToPILImage()(img.cpu())
                    imgs_aug.append(augmented_transform(pil_img))
                imgs_aug = torch.stack(imgs_aug).to(device)
                with torch.cuda.amp.autocast():
                    tta_logits.append(classifier(encoder(imgs_aug)))
            avg_logits = torch.stack(tta_logits, dim=0).mean(dim=0)
            softmaxed = torch.softmax(avg_logits, dim=1)
            preds = torch.argmax(avg_logits, dim=1)
            for i, img_id in enumerate(ids):
                predictions[img_id.item()] = base_dataset.classes[preds[i].item()]
                probs[img_id.item()] = softmaxed[i].cpu().numpy()
    return predictions, probs

# ----------------------------------
# Final Prediction and Saving
# ----------------------------------
def predict_and_save(model_name, encoder, classifier):
    preds, prob = predict_tta(encoder, classifier, num_tta=5)
    df = pd.DataFrame(sorted(preds.items()), columns=["id", "label"])
    df.to_csv(f"{OUTPUT_DIR}/{model_name}_pred.csv", index=False)
    return preds, prob

# ----------------------------------
# Weighted Ensemble
# ----------------------------------
def ensemble_weighted(pred_dicts, val_scores, prob_dicts):
    weights = np.array(val_scores)
    weights = weights / np.sum(weights)
    final_probs = defaultdict(lambda: np.zeros(num_classes))
    for weight, prob in zip(weights, prob_dicts):
        for k, v in prob.items():
            final_probs[k] += weight * v
    final = {k: base_dataset.classes[np.argmax(v)] for k, v in final_probs.items()}
    pd.DataFrame(sorted(final.items()), columns=['id', 'label']).to_csv(f"{OUTPUT_DIR}/weighted_ensemble.csv", index=False)

# ----------------------------------
# Main Routine: Cross-Validation & Final Training
# ----------------------------------
cv_val_scores = {}  # Store average CV accuracy per model type
cv_models = {}      # Store CV models per model type
model_names = ["resnet50", "vgg16", "efficientnet_b0"]

for model_name in model_names:
    print(f"\nTraining with cross-validation for model: {model_name}")
    avg_val_acc, models_cv = train_cv(model_name)
    cv_val_scores[model_name] = avg_val_acc
    cv_models[model_name] = models_cv

all_preds, all_probs, final_val_scores = [], [], []
for model_name in model_names:
    print(f"\nTraining full model for final prediction: {model_name}")
    encoder, _ = train_and_finetune(model_name)  # Train full model on full training data
    # For final prediction, create a fresh classifier and wrap it in DataParallel if needed
    feat_dim = get_model(model_name)[2]
    classifier = nn.Linear(feat_dim, num_classes).to(device)
    classifier = maybe_data_parallel(classifier)
    pred, prob = predict_and_save(model_name, encoder, classifier)
    all_preds.append(pred)
    all_probs.append(prob)
    final_val_scores.append(cv_val_scores[model_name])

ensemble_weighted(all_preds, final_val_scores, all_probs)




Training with cross-validation for model: resnet50
Fold 1/3


[I 2025-04-14 20:35:57,814] A new study created in memory with name: no-name-863658da-f854-4583-a3b7-e8af27b3251d
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s] 
/tmp/ipykernel_31/574533270.py:160: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_31/574533270.py:167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[I 2025-04-14 20:37:58,137] Trial 0 finished with value: 3.341821437701583 and parameters: {'lr': 1.565727009481795e-05, 'optimizer': 'SGD'}. Best is trial 0 with value: 3.341821437701583.
[I 2025-04-14 20:39:42,504] Trial 1 finished with value: 3.085392912849784 and parameters: {'lr': 6.19154825898097e-

Best Params for fold 1: {'lr': 8.79381291035577e-05, 'optimizer': 'Adam'}


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_31/574533270.py:217: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
resnet50 Fold 1 Epoch 1:   0%|          | 0/64 [00:00<?, ?it/s]/tmp/ipykernel_31/574533270.py:222: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
resnet50 Fold 1 Epoch 1: 100%|██████████| 64/64 [00:51<00:00,  1.24it/s]


Fold 1 Epoch 1 Loss: 2.6126


...

efficientnet_b0 Fold 3 Epoch 9: 100%|██████████| 64/64 [00:52<00:00,  1.22it/s]


Fold 3 Epoch 9 Loss: 2.2952


efficientnet_b0 Fold 3 Epoch 10: 100%|██████████| 64/64 [00:51<00:00,  1.24it/s]


Fold 3 Epoch 10 Loss: 2.2713
Fold 3 Validation Accuracy: 0.3136

Training full model for final prediction: resnet50


NameError: name 'train_and_finetune' is not defined